In [0]:
# COMMAND ----------

from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_accounts"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_accounts"

S3_BASE_PATH = "s3://zubair-s3-demo/raw_dataset/aml"
S3_DELTA_PATH = f"{S3_BASE_PATH}/delta_tables"

SILVER_ACCOUNTS_PATH = f"{S3_DELTA_PATH}/silver_accounts"


# ============================================================
# 2. READ BRONZE ACCOUNTS
# ============================================================

bronze_accounts_df = spark.table(BRONZE_TABLE)

print(f"Bronze table : {BRONZE_TABLE}")
print(f"Record count : {bronze_accounts_df.count()}")

display(bronze_accounts_df.limit(10))

In [0]:
# COMMAND ----------

silver_accounts_df = (
    bronze_accounts_df
    .select(
        F.col("ACCOUNT_ID")
            .cast("long")
            .alias("account_id"),

        F.trim(
            F.col("CUSTOMER_ID")
        ).alias("customer_id"),

        F.col("INIT_BALANCE")
            .cast("double")
            .alias("init_balance"),

        F.upper(
            F.trim(F.col("COUNTRY"))
        ).alias("country"),

        F.upper(
            F.trim(F.col("ACCOUNT_TYPE"))
        ).alias("account_type"),

        F.col("IS_FRAUD")
            .cast("boolean")
            .alias("is_fraud"),

        F.col("TX_BEHAVIOR_ID")
            .cast("long")
            .alias("tx_behavior_id")
    )
)

display(silver_accounts_df.limit(10))

In [0]:
# COMMAND ----------

silver_accounts_df = (
    silver_accounts_df

    # Account ID is mandatory
    .filter(
        F.col("account_id").isNotNull()
    )

    # Customer ID is mandatory
    .filter(
        F.col("customer_id").isNotNull()
    )

    # Initial balance must exist
    .filter(
        F.col("init_balance").isNotNull()
    )

    # Balance cannot be negative
    .filter(
        F.col("init_balance") >= 0
    )
)

In [0]:
# COMMAND ----------

silver_accounts_df = (
    silver_accounts_df
    .dropDuplicates(["account_id"])
)

In [0]:
# COMMAND ----------

silver_accounts_df = (
    silver_accounts_df
    .withColumn(
        "_silver_processed_timestamp",
        F.current_timestamp()
    )
)

In [0]:
# COMMAND ----------

(
    silver_accounts_df.write
    .format("delta")
    .mode("overwrite")
    .option("path", SILVER_ACCOUNTS_PATH)
    .saveAsTable(SILVER_TABLE)
)

print("Silver Accounts table created successfully.")
print(f"Table      : {SILVER_TABLE}")
print(f"S3 location: {SILVER_ACCOUNTS_PATH}")